   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0


# Phase 1
- collect the excel file in the cloud (use tmux to let it run een if you disconnect from terminal)
- build a semantic search engine (vector database) with the code below

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import numpy as np


## this code is to retrieve my key, which is saved in colab
import os
from google.colab import userdata
api_key = userdata.get("PINECONE_KEY") # you have to make an account on pinecone.io (it's free)

pc = Pinecone(api_key=api_key)

index_name = "semantic-search"
if index_name not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)
model = SentenceTransformer("all-MiniLM-L6-v2")

# HERE ARE 10 DOCUMENTS, WHICH IN THIS EXAMPLE ARE FICTITIOUS ABSTRACTS OF 10 PAPERS
texts = [

# -------------------- Quantum Computing (5) --------------------

"""We investigate variational quantum algorithms for solving electronic structure problems on noisy intermediate-scale quantum devices. Our approach combines hardware-efficient ansätze with adaptive parameter optimization to reduce circuit depth while preserving expressivity. Experimental results on superconducting qubit backends demonstrate improved convergence stability compared to standard VQE implementations. We further analyze the impact of depolarizing noise and show that moderate error mitigation significantly enhances ground-state energy estimation accuracy.""",

"""tODAY i ATE AN ICE CREAM AND DRANK A COCA COLA""",

"""We present a hybrid quantum-classical algorithm for combinatorial optimization using quantum approximate optimization (QAOA). The method incorporates classical gradient-based refinement to stabilize parameter updates under realistic noise conditions. Benchmarks on Max-Cut instances demonstrate improved solution quality compared to fixed-parameter schedules. Our results highlight the tradeoff between circuit depth and robustness in near-term optimization tasks.""",

"""This paper introduces a quantum error mitigation technique based on zero-noise extrapolation combined with randomized compiling. We experimentally evaluate the method on a 16-qubit processor and observe consistent suppression of coherent gate errors. The proposed framework reduces bias in expectation value estimation without requiring full quantum error correction. These findings suggest practical pathways for enhancing reliability in NISQ-era experiments.""",

"""We develop a quantum machine learning framework for kernel-based classification using photonic qubits. By encoding classical data into entangled states and measuring overlap via interference circuits, we construct quantum feature maps with enhanced separability. Experimental validation on small datasets demonstrates improved classification margins relative to classical radial basis kernels. The approach offers insights into potential quantum advantage in supervised learning.""",

# -------------------- Etching / Microfabrication (5) --------------------

"""We examine anisotropic silicon etching using inductively coupled plasma (ICP) systems for high-aspect-ratio microstructures. By optimizing gas composition and RF power, we achieve vertical sidewalls with minimal footing effects. Surface characterization via scanning electron microscopy reveals uniform etch profiles across 200 mm wafers. The process demonstrates strong repeatability for MEMS fabrication.""",

"""This study investigates atomic layer etching (ALE) of aluminum oxide for advanced semiconductor pattern transfer. A cyclic fluorination and argon ion activation sequence enables sub-nanometer etch control per cycle. We evaluate selectivity relative to photoresist and silicon substrates and report consistent removal rates below 1 Å per cycle. The results highlight ALE as a promising technique for sub-5 nm device scaling.""",

"""We present a comparative analysis of wet chemical etching and plasma-based dry etching for gallium nitride devices. Wet etchants provide smooth surface morphology but limited anisotropy, while chlorine-based plasma etching yields superior pattern fidelity. Electrical characterization of fabricated devices indicates that surface damage is minimized under optimized plasma bias conditions. The study informs process selection for high-frequency GaN electronics.""",

"""Reactive ion etching (RIE) parameters were optimized to fabricate nanoscale trenches in silicon dioxide. By tuning chamber pressure and electrode bias, we control ion energy distribution and reduce micro-masking artifacts. Atomic force microscopy measurements confirm improved surface roughness compared to baseline conditions. The developed recipe supports advanced lithographic pattern transfer.""",

"""We analyze plasma-induced damage mechanisms during deep reactive ion etching (DRIE) of silicon. Time-resolved optical emission spectroscopy was used to monitor plasma species concentration during Bosch cycling. Results show that polymer deposition dynamics strongly influence sidewall passivation and etch uniformity. Process optimization leads to enhanced aspect ratio capability while maintaining structural integrity."""
]


vectors = model.encode(texts).tolist()

index.upsert([
    {"id": str(i), "values": vectors[i], "metadata": {"text": texts[i]}}
    for i in range(len(texts))
])



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


UpsertResponse(upserted_count=10, _response_info={'raw_headers': {'date': 'Thu, 19 Feb 2026 23:10:04 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '7', 'x-pinecone-request-logical-size': '19644', 'x-pinecone-request-latency-ms': '317', 'x-envoy-upstream-service-time': '240', 'x-pinecone-response-duration-ms': '319', 'grpc-status': '0', 'server': 'envoy'}})

# Phase 2
The input is the Excel file collected in phase 1. Phase 2 is interactive. Every time the user writes a query, compute the score (see below) that measures teh simiularity of the query to each abstract. This score is a new column for the Excel file and can be used to sort the results

In [ ]:


# QUERY PHASE 2: the score below is the relevance (similarity) of the query with each document
query_vec = model.encode(["hungry"]).tolist()[0]
results = index.query(vector=query_vec, top_k=3, include_metadata=True)

print(results)


QueryResponse(matches=[{'id': '1',
 'metadata': {'text': 'tODAY i ATE AN ICE CREAM AND DRANK A COCA COLA'},
 'score': 0.275099754,
 'values': []}, {'id': '0',
 'metadata': {'text': 'We investigate variational quantum algorithms for '
                      'solving electronic structure problems on noisy '
                      'intermediate-scale quantum devices. Our approach '
                      'combines hardware-efficient ansätze with adaptive '
                      'parameter optimization to reduce circuit depth while '
                      'preserving expressivity. Experimental results on '
                      'superconducting qubit backends demonstrate improved '
                      'convergence stability compared to standard VQE '
                      'implementations. We further analyze the impact of '
                      'depolarizing noise and show that moderate error '
                      'mitigation significantly enhances ground-state energy '
                 